In [1]:
!pip install -qU ultralytics opencv-python numpy pandas tqdm

In [2]:
import argparse
import sys
import time
from collections import deque
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

In [3]:
# ---------------------------------------------------------------------------
# COCO 17-keypoint indices (used by YOLO Pose)
# ---------------------------------------------------------------------------
KP = {
    "nose": 0,
    "left_eye": 1,
    "right_eye": 2,
    "left_ear": 3,
    "right_ear": 4,
    "left_shoulder": 5,
    "right_shoulder": 6,
    "left_elbow": 7,
    "right_elbow": 8,
    "left_wrist": 9,
    "right_wrist": 10,
    "left_hip": 11,
    "right_hip": 12,
    "left_knee": 13,
    "right_knee": 14,
    "left_ankle": 15,
    "right_ankle": 16,
}

# Approximate segment mass fractions (de Leva 1996, male)
SEGMENT_WEIGHTS = {
    "head":           ("nose",            0.0694),
    "trunk":          (("left_shoulder", "right_shoulder", "left_hip", "right_hip"), 0.4346),
    "left_upper_arm": ("left_elbow",      0.0271),
    "right_upper_arm":("right_elbow",     0.0271),
    "left_forearm":   ("left_wrist",      0.0162),
    "right_forearm":  ("right_wrist",     0.0162),
    "left_thigh":     ("left_knee",       0.1416),
    "right_thigh":    ("right_knee",      0.1416),
    "left_shank":     ("left_ankle",      0.0433),
    "right_shank":    ("right_ankle",     0.0433),
}

# Confidence threshold for a keypoint to be considered valid
CONF_THRESHOLD = 0.40

# How many past frames to keep in the balance history heatmap strip
HISTORY_FRAMES = 120          # ~4 s at 30 fps
HEATMAP_HEIGHT = 90           # px height of the heatmap panel
HEATMAP_LABEL_WIDTH = 60      # px for axis labels

# Smoothing window for COM trail
TRAIL_LEN = 15

# Input video path
VIDEO_PATH = "/content/Screen Recording 2026-02-22 094732.mp4"
OUTPUT_VIDEO = "/content/output_weight_balance.mp4"

MODEL_PATH = "/content/yolo11s-pose.pt"

OUTPUT_CSV = "/content/pose_features.csv"

In [4]:
# ---------------------------------------------------------------------------
# Colour palette
# ---------------------------------------------------------------------------
C = {
    "com":        (0,   220, 255),   # cyan
    "com_trail":  (0,   140, 200),
    "left":       (50,  200,  50),   # green  → left foot
    "right":      (50,   80, 220),   # blue   → right foot
    "neutral":    (200, 200, 200),
    "bg":         (15,   15,  15),
    "panel_bg":   (25,   25,  25),
    "text":       (240, 240, 240),
    "accent":     (255, 180,   0),
    "skeleton":   (100, 100, 255),
    "keypoint":   (255, 255,  80),
    "warning":    (255,  60,  60),
}

# COCO skeleton pairs for drawing
SKELETON_PAIRS = [
    (KP["left_shoulder"],  KP["right_shoulder"]),
    (KP["left_shoulder"],  KP["left_elbow"]),
    (KP["left_elbow"],     KP["left_wrist"]),
    (KP["right_shoulder"], KP["right_elbow"]),
    (KP["right_elbow"],    KP["right_wrist"]),
    (KP["left_shoulder"],  KP["left_hip"]),
    (KP["right_shoulder"], KP["right_hip"]),
    (KP["left_hip"],       KP["right_hip"]),
    (KP["left_hip"],       KP["left_knee"]),
    (KP["left_knee"],      KP["left_ankle"]),
    (KP["right_hip"],      KP["right_knee"]),
    (KP["right_knee"],     KP["right_ankle"]),
    (KP["nose"],           KP["left_shoulder"]),
    (KP["nose"],           KP["right_shoulder"]),
]

In [5]:
# ---------------------------------------------------------------------------
# Utility helpers
# ---------------------------------------------------------------------------

def put_text(img, text, pos, scale=0.55, color=C["text"], thickness=1,
             font=cv2.FONT_HERSHEY_SIMPLEX, bg=None):
    """Draw text with optional background rectangle."""
    (tw, th), bl = cv2.getTextSize(text, font, scale, thickness)
    x, y = pos
    if bg is not None:
        cv2.rectangle(img, (x - 3, y - th - 3), (x + tw + 3, y + bl + 1), bg, -1)
    cv2.putText(img, text, (x, y), font, scale, color, thickness, cv2.LINE_AA)


def draw_bar(img, x, y, w, h, frac, color_left, color_right, label_l="L", label_r="R"):
    """Draw a horizontal split bar (left fraction, right fraction)."""
    split = int(w * frac)
    cv2.rectangle(img, (x, y), (x + split, y + h), color_left, -1)
    cv2.rectangle(img, (x + split, y), (x + w, y + h), color_right, -1)
    cv2.rectangle(img, (x, y), (x + w, y + h), (80, 80, 80), 1)
    put_text(img, label_l, (x + 4, y + h - 6), 0.42, (255, 255, 255))
    put_text(img, label_r, (x + w - 20, y + h - 6), 0.42, (255, 255, 255))


def alpha_blend(base, overlay, alpha):
    """Blend overlay onto base in-place with given alpha (0–1)."""
    mask = np.full(base.shape, alpha)
    cv2.addWeighted(base, 1 - alpha, overlay, alpha, 0, dst=base)

In [6]:
# ---------------------------------------------------------------------------
# Core analysis functions
# ---------------------------------------------------------------------------

def extract_keypoints(result, frame_w, frame_h):
    """
    Extract the highest-confidence person's keypoints from a YOLO result.

    Returns:
        kps  : np.ndarray shape (17, 2) pixel coords  (NaN where invalid)
        confs: np.ndarray shape (17,)   confidence scores
    """
    if result.keypoints is None or len(result.keypoints.data) == 0:
        return None, None

    # Pick the detection with the highest mean keypoint confidence
    kp_data = result.keypoints.data.cpu().numpy()  # (N, 17, 3)  x, y, conf
    mean_confs = kp_data[:, :, 2].mean(axis=1)
    best = np.argmax(mean_confs)

    raw = kp_data[best]  # (17, 3)
    confs = raw[:, 2]
    kps = raw[:, :2].copy()
    # Invalidate low-confidence points
    kps[confs < CONF_THRESHOLD] = np.nan
    return kps, confs


def compute_com(kps, confs):
    """
    Compute a weighted Centre of Mass pixel coordinate.

    Uses body-segment mass fractions.  Segments whose representative
    keypoint(s) are invalid are skipped and weights renormalised.

    Returns (cx, cy) or None if insufficient data.
    """
    total_w = 0.0
    wx, wy = 0.0, 0.0

    for seg_name, (kp_ref, mass) in SEGMENT_WEIGHTS.items():
        if isinstance(kp_ref, str):
            # Single keypoint
            idx = KP[kp_ref]
            if np.isnan(kps[idx, 0]):
                continue
            px, py = kps[idx]
        else:
            # Average over multiple keypoints (trunk)
            pts = []
            for k in kp_ref:
                idx = KP[k]
                if not np.isnan(kps[idx, 0]):
                    pts.append(kps[idx])
            if len(pts) < 2:
                continue
            arr = np.array(pts)
            px, py = arr[:, 0].mean(), arr[:, 1].mean()

        wx += mass * px
        wy += mass * py
        total_w += mass

    if total_w < 0.30:   # less than 30 % of body recovered
        return None

    return wx / total_w, wy / total_w

In [7]:
def compute_weight_distribution(com, kps):
    """
    Estimate left/right weight fraction from COM horizontal projection
    relative to ankle positions.

    Returns (left_frac, right_frac)  (both in [0,1], sum ≈ 1)
    """
    l_ankle = kps[KP["left_ankle"]]
    r_ankle = kps[KP["right_ankle"]]

    valid_l = not np.isnan(l_ankle[0])
    valid_r = not np.isnan(r_ankle[0])

    if not valid_l and not valid_r:
        return 0.5, 0.5

    if valid_l and not valid_r:
        return 1.0, 0.0

    if valid_r and not valid_l:
        return 0.0, 1.0

    # Both ankles valid – linear interpolation between them
    lx, rx = l_ankle[0], r_ankle[0]
    com_x = com[0]

    # Handle side-on view: left ankle may be visually right of right ankle
    # We use x-coords as-is; ensure proper ordering
    min_x, max_x = min(lx, rx), max(lx, rx)
    span = max_x - min_x

    if span < 5:   # feet almost on same vertical line
        return 0.5, 0.5

    # Fraction of COM toward the foot that is visually to the LEFT of frame
    # (not necessarily anatomically left foot)
    t = np.clip((com_x - min_x) / span, 0.0, 1.0)

    # Map t to anatomical left/right weight
    # If left ankle is visually left (lx < rx): left = 1-t, right = t
    # If left ankle is visually right (lx > rx): left = t, right = 1-t
    if lx <= rx:
        left_frac = 1.0 - t
        right_frac = t
    else:
        left_frac = t
        right_frac = 1.0 - t

    return float(left_frac), float(right_frac)


def balance_label(left_frac):
    """Return a descriptive balance string."""
    if left_frac > 0.70:
        return "HEAVY LEFT", C["left"]
    elif left_frac > 0.55:
        return "LEFT", C["left"]
    elif left_frac < 0.30:
        return "HEAVY RIGHT", C["right"]
    elif left_frac < 0.45:
        return "RIGHT", C["right"]
    else:
        return "BALANCED", C["accent"]

In [8]:
# ---------------------------------------------------------------------------
# Drawing functions
# ---------------------------------------------------------------------------

def draw_skeleton(frame, kps, confs):
    """Draw skeleton lines and keypoint dots."""
    for p1, p2 in SKELETON_PAIRS:
        if np.isnan(kps[p1, 0]) or np.isnan(kps[p2, 0]):
            continue
        pt1 = (int(kps[p1, 0]), int(kps[p1, 1]))
        pt2 = (int(kps[p2, 0]), int(kps[p2, 1]))
        cv2.line(frame, pt1, pt2, C["skeleton"], 2, cv2.LINE_AA)

    for i in range(17):
        if np.isnan(kps[i, 0]):
            continue
        cx, cy = int(kps[i, 0]), int(kps[i, 1])
        cv2.circle(frame, (cx, cy), 4, C["keypoint"], -1, cv2.LINE_AA)
        cv2.circle(frame, (cx, cy), 4, (0, 0, 0), 1, cv2.LINE_AA)


def draw_com(frame, com, trail):
    """Draw COM point and its motion trail."""
    # Trail
    for j, pt in enumerate(trail):
        alpha_factor = (j + 1) / len(trail)
        r = max(2, int(6 * alpha_factor))
        color = tuple(int(c * alpha_factor) for c in C["com_trail"])
        cv2.circle(frame, (int(pt[0]), int(pt[1])), r, color, -1, cv2.LINE_AA)

    # Current COM
    cx, cy = int(com[0]), int(com[1])
    cv2.circle(frame, (cx, cy), 10, C["com"], -1, cv2.LINE_AA)
    cv2.circle(frame, (cx, cy), 10, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.line(frame, (cx - 14, cy), (cx + 14, cy), C["com"], 2, cv2.LINE_AA)
    cv2.line(frame, (cx, cy - 14), (cx, cy + 14), C["com"], 2, cv2.LINE_AA)
    put_text(frame, "COM", (cx + 12, cy - 8), 0.42, C["com"], 1, bg=(20, 20, 20))


def draw_foot_markers(frame, kps):
    """Highlight ankle positions with foot markers."""
    for name, color in [("left_ankle", C["left"]), ("right_ankle", C["right"])]:
        idx = KP[name]
        if np.isnan(kps[idx, 0]):
            continue
        fx, fy = int(kps[idx, 0]), int(kps[idx, 1])
        cv2.circle(frame, (fx, fy), 8, color, -1, cv2.LINE_AA)
        cv2.circle(frame, (fx, fy), 8, (255, 255, 255), 2, cv2.LINE_AA)


def draw_ground_projection(frame, com, kps, frame_h):
    """Draw vertical dotted line from COM to ground + foot span."""
    cx = int(com[0])
    l_ankle = kps[KP["left_ankle"]]
    r_ankle = kps[KP["right_ankle"]]

    ground_y = frame_h - 10
    # Use ankle y if available
    ys = []
    if not np.isnan(l_ankle[0]):
        ys.append(int(l_ankle[1]))
    if not np.isnan(r_ankle[0]):
        ys.append(int(r_ankle[1]))
    if ys:
        ground_y = max(ys) + 5

    # Dotted vertical line
    com_y = int(com[1])
    for y in range(com_y, ground_y, 8):
        cv2.circle(frame, (cx, y), 1, C["com"], -1)

    # Horizontal span between feet
    if not np.isnan(l_ankle[0]) and not np.isnan(r_ankle[0]):
        lx, ly = int(l_ankle[0]), int(l_ankle[1])
        rx, ry = int(r_ankle[0]), int(r_ankle[1])
        avg_y = (ly + ry) // 2 + 5
        cv2.line(frame, (lx, avg_y), (rx, avg_y), C["neutral"], 1, cv2.LINE_AA)
        cv2.circle(frame, (cx, avg_y), 4, C["com"], -1, cv2.LINE_AA)


def draw_analytics_panel(frame, com, left_frac, right_frac, frame_idx, fps):
    """Draw the side analytics panel (top-right corner)."""
    fh, fw = frame.shape[:2]
    panel_w, panel_h = 180, 100  # Adjusted panel height for better spacing

    px, py = fw - panel_w - 10, 10

    # Semi-transparent background
    overlay = frame.copy()
    cv2.rectangle(overlay, (px, py), (px + panel_w, py + panel_h), C["panel_bg"], -1)
    cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)

    cv2.rectangle(frame, (px, py), (px + panel_w, py + panel_h), (70, 70, 70), 1)

    # Title
    put_text(frame, "WEIGHT BALANCE", (px + 8, py + 22), 0.52, C["accent"], 1)

    # Left / Right %
    put_text(frame, f"Left   {left_frac*100:5.1f}%", (px + 8, py + 48), 0.50, C["left"], 1)
    put_text(frame, f"Right  {right_frac*100:5.1f}%", (px + 8, py + 70), 0.50, C["right"], 1)

    # Balance label
    label, lcolor = balance_label(left_frac)
    put_text(frame, label, (px + 8, py + 95), 0.55, lcolor, 2)

    # Split bar
    draw_bar(frame, px + 8, py + 80, panel_w - 16, 18, left_frac, C["left"], C["right"])

In [9]:
# ---------------------------------------------------------------------------
# Heatmap panel builder
# ---------------------------------------------------------------------------

class BalanceHeatmap:
    """
    Maintains a rolling history of left-weight fractions and renders
    a colour heatmap strip + time-series line at the bottom of the frame.
    """

    def __init__(self, max_frames=HISTORY_FRAMES, panel_h=HEATMAP_HEIGHT):
        self.history = deque(maxlen=max_frames)
        self.panel_h = panel_h
        self.max_frames = max_frames

    def update(self, left_frac):
        self.history.append(left_frac)

    def render(self, frame):
        """Append heatmap panel below frame (returns new taller frame)."""
        fh, fw = frame.shape[:2]
        ph = self.panel_h
        lw = HEATMAP_LABEL_WIDTH

        panel = np.full((ph, fw, 3), C["bg"], dtype=np.uint8)

        # Title
        cv2.putText(panel, "WEIGHT BALANCE HISTORY  (green=Left  blue=Right)",
                    (lw + 4, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.40,
                    (180, 180, 180), 1, cv2.LINE_AA)

        hm_x = lw
        hm_w = fw - lw - 4
        hm_y_top = 20
        hm_h = ph - hm_y_top - 4

        # Build heatmap row by row
        hist = list(self.history)
        n = len(hist)
        if n == 0:
            combined = np.zeros((1, fw, 3), dtype=np.uint8)
        else:
            # Map each frame's fraction to a colour strip
            strip = np.zeros((1, n, 3), dtype=np.uint8)
            for i, lf in enumerate(hist):
                # Colour: pure green (left=1) ←→ pure blue (right=1)
                g = int(200 * lf)
                b = int(200 * (1 - lf))
                r = 0
                # Highlight near-balanced in yellow
                if 0.45 < lf < 0.55:
                    r, g, b = 180, 160, 0
                strip[0, i] = (b, g, r)

            # Scale strip to heatmap width
            strip_scaled = cv2.resize(strip, (hm_w, hm_h),
                                      interpolation=cv2.INTER_NEAREST)
            panel[hm_y_top:hm_y_top + hm_h, hm_x:hm_x + hm_w] = strip_scaled

        # Overlay line chart of left_frac
        if len(hist) > 1:
            pts = []
            for i, lf in enumerate(hist):
                sx = hm_x + int(i / max(len(hist) - 1, 1) * (hm_w - 1))
                sy = hm_y_top + int((1 - lf) * (hm_h - 1))
                pts.append((sx, sy))
            for i in range(1, len(pts)):
                cv2.line(panel, pts[i - 1], pts[i], (255, 255, 255), 1, cv2.LINE_AA)

        # Balance centre line
        cy = hm_y_top + hm_h // 2
        cv2.line(panel, (hm_x, cy), (hm_x + hm_w, cy), (255, 200, 0), 1)

        # Axis labels
        cv2.putText(panel, "L", (4, hm_y_top + 10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, C["left"], 1, cv2.LINE_AA)
        cv2.putText(panel, "R", (4, hm_y_top + hm_h - 2), cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, C["right"], 1, cv2.LINE_AA)
        cv2.putText(panel, "B", (4, cy + 5), cv2.FONT_HERSHEY_SIMPLEX,
                    0.38, (200, 180, 0), 1, cv2.LINE_AA)

        # Current frame cursor
        if n > 0:
            cx_cur = hm_x + hm_w - 1
            cv2.line(panel, (cx_cur, hm_y_top), (cx_cur, hm_y_top + hm_h),
                     (255, 255, 255), 1)

        # Border
        cv2.rectangle(panel, (hm_x, hm_y_top), (hm_x + hm_w - 1, hm_y_top + hm_h - 1),
                      (80, 80, 80), 1)

        return np.vstack([frame, panel])

In [10]:
# ---------------------------------------------------------------------------
# Main processing pipeline
# ---------------------------------------------------------------------------

def process_video(input_path: str, output_path: str, model_name: str,
                  conf: float, target_person_id: int,
                  show_skeleton: bool, show_trail: bool) -> pd.DataFrame:

    print(f"\n{'='*60}")
    print(f"  Cricket Pose & Weight Balance Analyser")
    print(f"{'='*60}")
    print(f"  Input  : {input_path}")
    print(f"  Output : {output_path}")
    print(f"  Model  : {model_name}")
    print(f"{'='*60}\n")

    # ── Load model ────────────────────────────────────────────────────────
    print("Loading YOLO Pose model …")
    model = YOLO(model_name)

    # ── Open video ────────────────────────────────────────────────────────
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        sys.exit(f"ERROR: Cannot open video '{input_path}'")

    fw   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Output height = frame height + heatmap panel
    out_h = fh + HEATMAP_HEIGHT
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (fw, out_h))

    print(f"  Video  : {fw}×{fh} @ {fps:.1f} fps  ({total_frames} frames)")
    print()

    # ── State ────────────────────────────────────────────────────────────
    com_trail = deque(maxlen=TRAIL_LEN)
    heatmap   = BalanceHeatmap(max_frames=HISTORY_FRAMES)

    records = []     # for CSV export
    frame_idx = 0
    t_start = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # ── Inference ────────────────────────────────────────────────────
        results = model(frame, conf=conf, verbose=False)
        result  = results[0]

        kps, confs = extract_keypoints(result, fw, fh)

        com = None
        left_frac = right_frac = 0.5

        if kps is not None:
            com = compute_com(kps, confs)
            if com is not None:
                left_frac, right_frac = compute_weight_distribution(com, kps)
                com_trail.append(com)

        # ── Draw ─────────────────────────────────────────────────────────
        if show_skeleton and kps is not None:
            draw_skeleton(frame, kps, confs)

        if kps is not None:
            draw_foot_markers(frame, kps)

        if com is not None:
            if show_trail:
                draw_com(frame, com, list(com_trail)[:-1])
            else:
                draw_com(frame, com, [])
            draw_ground_projection(frame, com, kps, fh)

        draw_analytics_panel(frame, com if com else (0, 0),
                             left_frac, right_frac, frame_idx, fps)

        # Pose-not-detected warning
        if kps is None or com is None:
            put_text(frame, "⚠  NO POSE DETECTED",
                     (10, fh - 20), 0.65, C["warning"], 2, bg=(40, 0, 0))

        # ── Heatmap ──────────────────────────────────────────────────────
        heatmap.update(left_frac)
        combined = heatmap.render(frame)

        out.write(combined)

        # ── Record ───────────────────────────────────────────────────────
        records.append({
            "frame":       frame_idx,
            "time_s":      round(frame_idx / fps, 4),
            "com_x":       round(com[0], 2) if com else None,
            "com_y":       round(com[1], 2) if com else None,
            "left_frac":   round(left_frac, 4),
            "right_frac":  round(right_frac, 4),
            "balance":     balance_label(left_frac)[0],
        })

        # ── Progress ─────────────────────────────────────────────────────
        frame_idx += 1
        if frame_idx % 30 == 0 or frame_idx == 1:
            elapsed = time.time() - t_start
            pct = frame_idx / max(total_frames, 1) * 100
            print(f"  Frame {frame_idx:5d}/{total_frames}  ({pct:5.1f}%)  "
                  f"  left={left_frac*100:5.1f}%  right={right_frac*100:5.1f}%  "
                  f"  elapsed {elapsed:.1f}s")

    cap.release()
    out.release()

    print(f"\n✓ Saved processed video → {output_path}")

    df = pd.DataFrame(records)
    return df

In [11]:
# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def parse_args():
    p = argparse.ArgumentParser(
        description="Cricket batsman pose estimation & weight balance analysis")
    p.add_argument("--input",  "-i", required=True,  help="Input video path")
    p.add_argument("--output", "-o", default=None,   help="Output video path (default: <input>_analyzed.mp4)")
    p.add_argument("--model",  "-m", default="yolo11x-pose.pt",
                   help="YOLO Pose model (default: yolo11x-pose.pt)")
    p.add_argument("--conf",   "-c", type=float, default=0.35,
                   help="Detection confidence threshold (default: 0.35)")
    p.add_argument("--person-id", type=int, default=0,
                   help="Index of person to track when multiple detected (default: 0)")
    p.add_argument("--no-skeleton", action="store_true",
                   help="Disable skeleton overlay")
    p.add_argument("--no-trail", action="store_true",
                   help="Disable COM motion trail")
    p.add_argument("--csv",    default=None,
                   help="Path to save per-frame analytics CSV")
    return p.parse_args()

In [12]:
def main():
    # Store original sys.argv
    original_sys_argv = sys.argv[:]

    # Simulate command-line arguments for Colab execution
    # Using variables already available from previous cells
    sys.argv = [
        'colab_script_name.py', # The first element is typically the script name
        '--input', VIDEO_PATH,
        '--output', OUTPUT_VIDEO,
        '--model', MODEL_PATH,
        '--csv', OUTPUT_CSV,
        '--conf', '0.4' # Use a default confidence threshold
    ]

    try:
        args = parse_args()
    finally:
        # Restore sys.argv to its original state after parsing
        sys.argv = original_sys_argv

    input_path = args.input
    if args.output is None:
        stem = Path(input_path).stem
        output_path = str(Path(input_path).parent / f"{stem}_analyzed.mp4")
    else:
        output_path = args.output

    df = process_video(
        input_path    = input_path,
        output_path   = output_path,
        model_name    = args.model,
        conf          = args.conf,
        target_person_id = args.person_id,
        show_skeleton = not args.no_skeleton,
        show_trail     = not args.no_trail,
    )

    # ── Summary stats ────────────────────────────────────────────────────
    print("\n── Session Summary ──────────────────────────────────────────")
    detected = df.dropna(subset=["com_x"])
    print(f"  Frames processed : {len(df)}")
    print(f"  Pose detected    : {len(detected)} ({len(detected)/max(len(df),1)*100:.1f}%)者に教えてください。")
    if len(detected):
        print(f"  Avg left  weight : {detected['left_frac'].mean()*100:.1f}%")
        print(f"  Avg right weight : {detected['right_frac'].mean()*100:.1f}%")
        print(f"  Balance spread   : {detected['left_frac'].std()*100:.2f}% σ")
        print(f"  Balance labels:\n{detected['balance'].value_counts().to_string()}")

    # ── Save CSV ─────────────────────────────────────────────────────────
    csv_path = args.csv
    if csv_path is None:
        csv_path = str(Path(output_path).with_suffix(".csv"))
    df.to_csv(csv_path, index=False)
    print(f"\n✓ Saved analytics CSV  → {csv_path}")
    print("Done.\n")


main()


  Cricket Pose & Weight Balance Analyser
  Input  : /content/Screen Recording 2026-02-22 094732.mp4
  Output : /content/output_weight_balance.mp4
  Model  : /content/yolo11s-pose.pt

Loading YOLO Pose model …
  Video  : 280×388 @ 30.0 fps  (203 frames)

  Frame     1/203  (  0.5%)    left= 46.0%  right= 54.0%    elapsed 1.3s
  Frame    30/203  ( 14.8%)    left= 52.4%  right= 47.6%    elapsed 2.7s
  Frame    60/203  ( 29.6%)    left= 51.9%  right= 48.1%    elapsed 4.6s
  Frame    90/203  ( 44.3%)    left= 53.6%  right= 46.4%    elapsed 5.6s
  Frame   120/203  ( 59.1%)    left= 57.4%  right= 42.6%    elapsed 6.7s
  Frame   150/203  ( 73.9%)    left= 83.3%  right= 16.7%    elapsed 7.8s
  Frame   180/203  ( 88.7%)    left=100.0%  right=  0.0%    elapsed 9.0s

✓ Saved processed video → /content/output_weight_balance.mp4

── Session Summary ──────────────────────────────────────────
  Frames processed : 203
  Pose detected    : 202 (99.5%)者に教えてください。
  Avg left  weight : 58.4%
  Avg right we